In [2]:
import os
import glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, Input, mixed_precision
from sklearn.model_selection import train_test_split
import cv2
import keras
print("library loaded successfully")
print("*"*100)

library loaded successfully
****************************************************************************************************


In [3]:
DATA_PATH = '/mnt/c/development/Thesis/PolypSegmentationBasedClassification/DataSets/PolypGen2021_MultiCenterData_v2'
POSITIVE_DIR = os.path.join(DATA_PATH, 'imagesAll_positive') # For Polyp classification
NEGATIVE_DIR = os.path.join(DATA_PATH, 'sequenceData/negativeOnly')  # For non polyp classification
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/lr-ynet.keras"

In [4]:
def process_path_classification(img_path, label):
    """
    Preprocesses input tensors for Classification Dataset.
    Applies identical CLAHE and Sharpening to match the segmentation pipeline.
    """
    def _read_and_process_clf(img_p):
        img_p_str = img_p.numpy().decode('utf-8') if hasattr(img_p, 'numpy') else img_p.decode('utf-8')
        img = cv2.imread(img_p_str)
        if img is None:
            raise FileNotFoundError(f"❌ Image not found at path: {img_p_str}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        
        # ─── STEP 1: CONTRAST ENHANCEMENT (CLAHE) ───
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(lab)
        
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        cl = clahe.apply(l_channel)
        
        limg = cv2.merge((cl, a_channel, b_channel))
        enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)
        median_blur = cv2.medianBlur(enhanced_img, 3) 
        
        # ─── STEP 2: MILD SHARPENING ───
        sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
        
        # ─── STEP 3: NORMALIZATION ───
        normalized_img = sharpened / 255.0
        return normalized_img.astype(np.float32)
    [img_tensor] = tf.py_function(
        _read_and_process_clf, 
        [img_path], 
        [tf.float32]
    )

    img_tensor.set_shape([IMG_SIZE, IMG_SIZE, 3])
    
    return img_tensor, label



def load_classification_paths():
    """Compiles absolute paths for global positive and artifact-heavy negative sequences."""
    valid_exts = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.PNG', '*.JPG', '*.JPEG')
    pos_files, neg_files = [], []
    
    # 1. Gather all positive paths
    for ext in valid_exts:
        pos_files.extend(glob.glob(os.path.join(POSITIVE_DIR, "**", ext), recursive=True))
        
    # 2. Gather all negative paths from seq1 to seq13
    for i in range(1, 14):
        target_folder = os.path.join(NEGATIVE_DIR, f"seq{i}_neg")
        if os.path.exists(target_folder):
            for ext in valid_exts:
                neg_files.extend(glob.glob(os.path.join(target_folder, "**", ext), recursive=True))
                
    # Filter to ensure only valid files are kept
    pos_files = [f for f in pos_files if os.path.isfile(f)]
    neg_files = [f for f in neg_files if os.path.isfile(f)]
    
    # ─── 🚀 🆕 STEP 3: STRICT 50% - 50% BALANCING LOGIC ───
    min_samples = min(len(pos_files), len(neg_files))
    print(f"\n📊 Raw Counts -> Polyp: {len(pos_files)} | Non-Polyp: {len(neg_files)}")
    print(f"⚖️ Balancing Dataset at Sweet Spot: {min_samples} samples each (Total: {min_samples * 2})")
    
    # Randomly shuffle before trimming to ensure diversity
    np.random.seed(42)
    np.random.shuffle(pos_files)
    np.random.shuffle(neg_files)
    
    # Slice both lists to have the exact same size
    pos_files = pos_files[:min_samples]
    neg_files = neg_files[:min_samples]
    print("positive files", len(pos_files), "neg files", len(neg_files))
    # ──────────────────────────────────────────────────────
    
    file_paths = pos_files + neg_files
    labels = [1.0] * len(pos_files) + [0.0] * len(neg_files)
    
    return np.array(file_paths), np.array(labels, dtype=np.float32)

In [5]:
@tf.keras.utils.register_keras_serializable()
class KerasPCCLayer(layers.Layer):
    """Natively computes spatial Pearson Correlation Coefficients within the network graph."""
    def __init__(self, **kwargs):
        super(KerasPCCLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # We used epsilon sothat standard deviation value cannot be zero, otherwise code will crash
        epsilon = 1e-8
        # finds the average standard deviaiton
        mean = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        std = tf.math.reduce_std(inputs, axis=-1, keepdims=True) + epsilon
        # data normalization or (Z-Score)
        norm_inputs = (inputs - mean) / std
        pcc_matrix = tf.matmul(tf.expand_dims(norm_inputs, -1), tf.expand_dims(norm_inputs, 1))
        return layers.Flatten()(pcc_matrix)

In [6]:
@keras.saving.register_keras_serializable()
def dice_coef(y_true, y_pred, smooth=1.0):
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


In [7]:
@keras.saving.register_keras_serializable()
def iou_metric(y_true, y_pred):
    y_true_f = tf.cast(y_true, tf.float32)
    y_pred_f = tf.cast(y_pred > 0.5, tf.float32)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + 1.0) / (union + 1.0)

In [8]:
def bce_dice_loss(y_true, y_pred):
    bce = tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))
    return bce + dice_loss(y_true, y_pred)

In [9]:
def focal_tversky_loss(y_true, y_pred, alpha=0.7, beta=0.3, gamma=0.75):
    """
    Focal Tversky Loss for imbalanced medical image segmentation.
    alpha: controls penalty for False Negatives (FN) -> keeping it higher (0.7) focuses on small polyps.
    beta: controls penalty for False Positives (FP).
    """
    y_true_f = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
    y_pred_f = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)
    
    # Calculate True Positives, False Negatives, and False Positives
    tp = tf.reduce_sum(y_true_f * y_pred_f)
    fn = tf.reduce_sum(y_true_f * (1.0 - y_pred_f))
    fp = tf.reduce_sum((1.0 - y_true_f) * y_pred_f)
    
    # Tversky Index
    smooth = 1e-6
    tversky_index = (tp + smooth) / (tp + alpha * fn + beta * fp + smooth)
    
    # Focal Tversky Loss
    return tf.math.pow((1.0 - tversky_index), gamma)

# ridge classifier

In [10]:
import os
import gc
import cv2
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

print("\n======================================================")
print(" [ABLATION] RIDGE CLASSIFIER ENGINE")
print("=======================================================")
# ─────────────────────────────────────────────────────────────────────────────
# 🛠️ STAGE 1: LOAD PATHS AND DEFINE DATA SPLIT (FIXED VARIABLES)
# ─────────────────────────────────────────────────────────────────────────────
clf_imgs, clf_labels = load_classification_paths()
indices = np.arange(len(clf_imgs))
np.random.seed(42)
np.random.shuffle(indices)
clf_imgs, clf_labels = clf_imgs[indices], clf_labels[indices]

c_split = int(0.8 * len(clf_imgs))

train_imgs_paths = clf_imgs[:c_split]
train_lbls_raw = clf_labels[:c_split]
val_imgs_paths = clf_imgs[c_split:]
val_lbls_raw = clf_labels[c_split:]

IMG_SIZE = 256

# Path to your trained segmentation/classification backbone (.keras)
MODEL_SAVE_PATH = "/mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras"

# ─────────────────────────────────────────────────────────────────────────────
# 🧠 STAGE 2: LOAD MODEL & EXTRACT EXTENDED EMBEDDINGS FROM PCC/GAP LAYER
# ─────────────────────────────────────────────────────────────────────────────
print(f"🔄 Fetching Weights Safely: {MODEL_SAVE_PATH}")
base_model = tf.keras.models.load_model(MODEL_SAVE_PATH, custom_objects={
    'KerasPCCLayer': KerasPCCLayer, 'dice_coef': dice_coef,
    'iou_metric': iou_metric, 'bce_dice_loss': bce_dice_loss,
    'focal_tversky_loss': focal_tversky_loss
}, compile=False)


try:
    target_layer_name = 'clf_pcc_layer'

    _ = base_model.get_layer(target_layer_name)
    print(f"🎯 [SUCCESS] Targeted your Custom PCC Layer: '{target_layer_name}' for feature harvesting.")
except Exception as e:
    target_layer_name = 'clf_gap'
    print(f"⚠️ Layer fallback to: '{target_layer_name}' due to: {e}")

feature_extractor = tf.keras.Model(
    inputs=base_model.input,
    outputs=base_model.get_layer(target_layer_name).output
)

# ─────────────────────────────────────────────────────────────────────────────
# 🏃‍♂️ STAGE 3: RAW PYTHON FEATURE HARVESTER (With CLAHE & Sharpening Alignment)
# ─────────────────────────────────────────────────────────────────────────────
def pure_python_feature_extractor(img_paths, labels_raw):
    extracted_features = []
    final_labels = []

    total_samples = len(img_paths)
    print(f"⏳ Harvesting embeddings from {total_samples} samples...")

    for idx in range(total_samples):
        try:
            img = cv2.imread(img_paths[idx])
            if img is None:
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))

            lab = cv2.cvtColor(img_res, cv2.COLOR_RGB2LAB)
            l_channel, a_channel, b_channel = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            cl = clahe.apply(l_channel)
            limg = cv2.merge((cl, a_channel, b_channel))
            enhanced_img = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

            median_blur = cv2.medianBlur(enhanced_img, 3)
            sharpened = cv2.addWeighted(enhanced_img, 1.5, median_blur, -0.5, 0)
            inp = np.expand_dims(sharpened / 255.0, axis=0).astype(np.float32)

            feats = feature_extractor(inp, training=False)
            feats_flat = tf.reshape(feats, [tf.shape(feats)[0], -1]).numpy().flatten().astype(np.float32)

            lbl = labels_raw[idx]
            if isinstance(lbl, (list, tuple, np.ndarray)):
                binary_label = lbl[-1] if len(lbl) > 1 else lbl[0]
            else:
                binary_label = lbl

            if hasattr(binary_label, 'numpy'):
                binary_label = binary_label.numpy()
            if isinstance(binary_label, (np.ndarray, list)) and len(binary_label) == 1:
                binary_label = binary_label[0]

            extracted_features.append(feats_flat)
            final_labels.append(int(binary_label))

        except Exception as e:
            continue

        if idx % 100 == 0:
            gc.collect()

    return np.array(extracted_features, dtype=np.float32), np.array(final_labels, dtype=np.int32)

print("\n🎬 Extracting Training Features...")
X_train_flat, y_train = pure_python_feature_extractor(train_imgs_paths, train_lbls_raw)

print("\n🎬 Extracting Validation Features...")
X_val_flat, y_val = pure_python_feature_extractor(val_imgs_paths, val_lbls_raw)

y_train = y_train.ravel()
y_val = y_val.ravel()

print(f"\n✅ Feature Extraction Complete!")
print(f" -> Train Labels Distribution (0: Normal / 1: Polyp): {np.bincount(y_train)}")
print(f" -> Val Labels Distribution   (0: Normal / 1: Polyp): {np.bincount(y_val)}")

del base_model
del feature_extractor
tf.keras.backend.clear_session()
gc.collect()

# ─────────────────────────────────────────────────────────────────────────────
# 📉 STAGE 4: FEATURE SELECTION & ROBUST SCALE (identical to ELM/LR/SVM pipeline)
# ─────────────────────────────────────────────────────────────────────────────
print("\n📉 Step A: Executing Feature Selection via SelectKBest (ANOVA F-Classifier)...")

max_k = min(300, X_train_flat.shape[1])
selector = SelectKBest(score_func=f_classif, k=max_k)
X_train_selected = selector.fit_transform(X_train_flat, y_train).astype(np.float32)
X_val_selected = selector.transform(X_val_flat).astype(np.float32)

del X_train_flat
del X_val_flat
gc.collect()

print("⚖️ Step B: Applying Robust Scaling (Outlier Proof)...")

scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_selected).astype(np.float32)
X_val_scaled = scaler.transform(X_val_selected).astype(np.float32)

del X_train_selected
del X_val_selected
gc.collect()

# ─────────────────────────────────────────────────────────────────────────────
# 🧮 STAGE 5: RIDGE CLASSIFIER (replaces Linear SVM / Logistic Regression / Ensemble ELM)
# ─────────────────────────────────────────────────────────────────────────────
print("\n🧮 Training Ridge Classifier on selected + scaled features...")

# RidgeClassifier fits a linear model on {-1, +1} targets via closed-form
# least-squares (conceptually the closest of all your ablation rows to the
# ELM's own ridge-regularized closed-form readout — but WITHOUT the random
# tanh projection layer, i.e. it's a direct linear ridge fit on the scaled
# features themselves). Like LinearSVC, it has no predict_proba, only
# decision_function, so it's wrapped the same way in CalibratedClassifierCV.
base_ridge = RidgeClassifier(
    alpha=1.0,                 # L2 regularization strength (plays the role of your ELM lambdas)
    class_weight='balanced',   # same imbalance handling as LR/SVM runs
    random_state=42
)

ridge_clf = CalibratedClassifierCV(base_ridge, method='sigmoid', cv=5)
ridge_clf.fit(X_train_scaled, y_train)

print("✓ Ridge Classifier (calibrated) trained. Sweeping anti-bias thresholds...")

# ─────────────────────────────────────────────────────────────────────────────
# 🎯 STAGE 6: EVALUATION WITH SENSITIVITY-SPECIFICITY EQUILIBRIUM (G-MEAN)
# ─────────────────────────────────────────────────────────────────────────────
def evaluate_ridge_classifier(X_matrix, y_true, set_name="Validation"):
    final_probs = ridge_clf.predict_proba(X_matrix)[:, 1]

    best_thresh = 0.5
    best_gmean = 0
    best_preds = np.zeros_like(y_true)

    for thresh in np.arange(0.05, 0.95, 0.02):
        current_preds = (final_probs > thresh).astype(int)

        tp = np.sum((current_preds == 1) & (y_true == 1))
        fp = np.sum((current_preds == 1) & (y_true == 0))
        fn = np.sum((current_preds == 0) & (y_true == 1))
        tn = np.sum((current_preds == 0) & (y_true == 0))

        sensitivity = tp / (tp + fn + 1e-7)
        specificity = tn / (tn + fp + 1e-7)

        gmean = np.sqrt(sensitivity * specificity)

        if gmean > best_gmean:
            best_gmean = gmean
            best_thresh = thresh
            best_preds = current_preds

    tp = np.sum((best_preds == 1) & (y_true == 1))
    fp = np.sum((best_preds == 1) & (y_true == 0))
    fn = np.sum((best_preds == 0) & (y_true == 1))
    tn = np.sum((best_preds == 0) & (y_true == 0))

    acc = np.mean(best_preds == y_true)
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    specificity = tn / (tn + fp + 1e-7)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-7)
    roc_auc = roc_auc_score(y_true, final_probs)
    prec_vals, rec_vals, _ = precision_recall_curve(y_true, final_probs)
    pr_auc = auc(rec_vals, prec_vals)

    print(f"\n🎯 [{set_name} Anti-Bias Results (Optimal Threshold={best_thresh:.2f})]:")
    print(f"    Accuracy    : {acc*100:.2f}%")
    print(f"    Precision   : {precision*100:.2f}%")
    print(f"    Recall(Sens): {recall*100:.2f}%")
    print(f"    Specificity : {specificity*100:.2f}%  <--- (High value means Normal images are safe!)")
    print(f"    F1-Score    : {f1*100:.2f}%")
    print(f"    G-Mean      : {best_gmean*100:.2f}%")
    print(f"    ROC-AUC     : {roc_auc*100:.2f}% 🎯")
    print(f"    PR-AUC      : {pr_auc*100:.2f}% 📈")

    return {
        "accuracy": acc, "precision": precision, "recall": recall,
        "specificity": specificity, "f1": f1, "g_mean": best_gmean,
        "roc_auc": roc_auc, "pr_auc": pr_auc, "threshold": best_thresh
    }

train_metrics = evaluate_ridge_classifier(X_train_scaled, y_train, set_name="Training")
val_metrics = evaluate_ridge_classifier(X_val_scaled, y_val, set_name="Validation")

# ─────────────────────────────────────────────────────────────────────────────
# 💾 STAGE 7: SAVE CLASSIFIER ARTIFACTS (for downstream inference)
# ─────────────────────────────────────────────────────────────────────────────
import joblib

RIDGE_ARTIFACT_PATH = "ridge_classifier_artifacts.joblib"

print(f"\n💾 Saving Ridge Classifier artifacts to: {RIDGE_ARTIFACT_PATH}")
joblib.dump({
    "calibrated_ridge": ridge_clf,               # full fitted CalibratedClassifierCV (needed for predict_proba)
    "selector_support": selector.get_support(),   # boolean mask over original 1024-dim features
    "scaler_center": scaler.center_.astype(np.float32),
    "scaler_scale": scaler.scale_.astype(np.float32),
    "decision_threshold": np.float32(val_metrics["threshold"]),
    "target_layer_name": target_layer_name,
    "img_size": IMG_SIZE,
}, RIDGE_ARTIFACT_PATH)
print("✓ Saved: calibrated_ridge, selector_support, scaler_center, scaler_scale, decision_threshold, target_layer_name, img_size")


 [ABLATION] RIDGE CLASSIFIER ENGINE

📊 Raw Counts -> Polyp: 3762 | Non-Polyp: 2520
⚖️ Balancing Dataset at Sweet Spot: 2520 samples each (Total: 5040)
positive files 2520 neg files 2520
🔄 Fetching Weights Safely: /mnt/c/development/Thesis/PolypSegmentationBasedClassification/models/SC/tr-elm/thesis_v3_sequential_v_eval_data_driven_v2.keras


W0000 00:00:1788497893.436643  149796 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1788497893.437059  149796 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


🎯 [SUCCESS] Targeted your Custom PCC Layer: 'clf_pcc_layer' for feature harvesting.

🎬 Extracting Training Features...
⏳ Harvesting embeddings from 4032 samples...


I0000 00:00:1788497896.977603  149796 cuda_dnn.cc:461] Loaded cuDNN version 92101



🎬 Extracting Validation Features...
⏳ Harvesting embeddings from 1008 samples...

✅ Feature Extraction Complete!
 -> Train Labels Distribution (0: Normal / 1: Polyp): [1987 2045]
 -> Val Labels Distribution   (0: Normal / 1: Polyp): [533 475]

📉 Step A: Executing Feature Selection via SelectKBest (ANOVA F-Classifier)...
⚖️ Step B: Applying Robust Scaling (Outlier Proof)...

🧮 Training Ridge Classifier on selected + scaled features...
✓ Ridge Classifier (calibrated) trained. Sweeping anti-bias thresholds...

🎯 [Training Anti-Bias Results (Optimal Threshold=0.49)]:
    Accuracy    : 93.68%
    Precision   : 94.31%
    Recall(Sens): 93.15%
    Specificity : 94.21%  <--- (High value means Normal images are safe!)
    F1-Score    : 93.73%
    G-Mean      : 93.68%
    ROC-AUC     : 97.98% 🎯
    PR-AUC      : 98.36% 📈

🎯 [Validation Anti-Bias Results (Optimal Threshold=0.57)]:
    Accuracy    : 93.55%
    Precision   : 94.57%
    Recall(Sens): 91.58%
    Specificity : 95.31%  <--- (High valu